In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, roc_curve
import pickle
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("PART B - FEATURE ENGINEERING, MODELING & PREDICTION")
print("="*70)

# LOAD THE CLEANED DATASET FROM PART A
df = pd.read_csv('Ayebare.csv')
print(f"\nLoaded Ayebare.csv: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Target variable (success): {df['success'].value_counts().to_dict()}")

# ============================================================
# QUESTION 1: FEATURE ENGINEERING [10 MARKS]
# ============================================================

print("\n" + "="*70)
print("QUESTION 1: FEATURE ENGINEERING [10 MARKS]")
print("="*70)

print("\nSTEP 1: Identify and Select Features")
print("-" * 70)

feature_engineering_doc = """
FEATURE ENGINEERING PLAN:

1. NUMERICAL FEATURES (Direct Use):
   ─────────────────────────────────
   
   • budget: Production investment in millions USD
     Rationale: Higher budgets enable larger marketing and production scope
     
   • duration: Runtime in minutes
     Rationale: Affects audience reach (family films shorter, epics longer)
     Range: 80-180 minutes is optimal
     
   • imdb_score: Critical reception on 1-10 scale
     Rationale: Direct proxy for critical acclaim and quality perception
     
   • num_voted_users: Number of IMDB votes
     Rationale: Indicates audience engagement and film reach
     
   • cast_total_facebook_likes: Combined actor popularity
     Rationale: Star power drives box office attendance
     
   • director_facebook_likes: Director popularity
     Rationale: Established directors have fan bases

2. CATEGORICAL FEATURES (One-Hot Encoding):
   ─────────────────────────────────────────
   
   • content_rating: MPAA rating (G, PG, PG-13, R, etc.)
     Rationale: Different ratings target different audiences
     Action: One-hot encode top 5 ratings, rest as 'Other'
     
   • genres: Film genre(s) (multiple genres per film)
     Rationale: Different genres have different success patterns
     Action: Split comma-separated genres and create binary features
     
   • color: Film format (Color vs Black & White)
     Rationale: Modern color films dominate box office

3. TEMPORAL FEATURES:
   ──────────────────
   
   • title_year: Release year
     Rationale: Captures inflation, market trends, audience preferences
     Action: Extract decade, calculate "years since release"
     
   • release_decade: Grouped year (1910s, 1920s, etc.)
     Rationale: Long-term trends in box office success

4. DERIVED FEATURES:
   ─────────────────
   
   • budget_to_runtime: budget / duration
     Rationale: Higher spend-per-minute indicates bigger production scope
     Formula: budget / duration
     Insight: Blockbuster films have high per-minute spend
     
   • budget_to_votes: budget / num_voted_users
     Rationale: Efficiency of audience engagement relative to investment
     
   • imdb_to_facebook_ratio: imdb_score / (cast_total_facebook_likes + 1)
     Rationale: Critical quality vs popularity (might diverge)
     
   • director_cast_ratio: director_facebook_likes / (cast_total_facebook_likes + 1)
     Rationale: Director prominence relative to cast
     
   • log_budget: log(budget + 1)
     Rationale: Normalize highly skewed budget distribution
     
   • log_votes: log(num_voted_users + 1)
     Rationale: Normalize engagement metrics

5. TEXT/SENTIMENT FEATURE:
   ──────────────────────
   
   • review_sentiment_score: Proportion of positive reviews
     Rationale: Audience sentiment predicts box office success

FEATURE SELECTION METHOD:
─────────────────────────
   1. Correlation analysis: Drop highly correlated features (r > 0.9)
   2. Feature importance: Use Random Forest to identify top features
   3. Final set: ~18-25 features for optimal modeling

EXPECTED OUTPUT:
   • ~20 total features (numerical + categorical + derived)
   • No missing values
   • Standardized scale for numerical features
"""

print(feature_engineering_doc)

# STEP 2: Create derived features
print("\nSTEP 2: Create Derived Features")
print("-" * 70)

df['log_budget'] = np.log1p(df['budget'])
df['log_votes'] = np.log1p(df['num_voted_users'])
df['budget_per_minute'] = df['budget'] / (df['duration'] + 1)
df['budget_per_vote'] = df['budget'] / (df['num_voted_users'] + 1)
df['log_cast_popularity'] = np.log1p(df['cast_total_facebook_likes'])
df['log_director_popularity'] = np.log1p(df['director_facebook_likes'])
df['cast_director_ratio'] = (df['director_facebook_likes'] + 1) / (df['cast_total_facebook_likes'] + 1)
df['quality_engagement'] = df['imdb_score'] * np.log1p(df['num_voted_users'])

print(" Created derived features:")
print("  • log_budget, log_votes, budget_per_minute, budget_per_vote")
print("  • log_cast_popularity, log_director_popularity")
print("  • cast_director_ratio, quality_engagement")

# STEP 3: Handle categorical features
print("\nSTEP 3: Process Categorical Features")
print("-" * 70)

# Content rating: Keep top 5, rest as 'Other'
top_ratings = df['content_rating'].value_counts().head(5).index.tolist()
df['content_rating_grouped'] = df['content_rating'].apply(
    lambda x: x if x in top_ratings else 'Other'
)

# One-hot encode content rating
rating_dummies = pd.get_dummies(df['content_rating_grouped'], prefix='rating')
df = pd.concat([df, rating_dummies], axis=1)
print(f" Content rating: One-hot encoded {len(rating_dummies.columns)} categories")

# STEP 4: Extract genres and create binary features
print("\nSTEP 4: Process Genres")
print("-" * 70)

# Split genres and create binary features for top genres
all_genres = df['genres'].str.split('|').explode().unique()
top_genres = df['genres'].str.split('|').explode().value_counts().head(8).index.tolist()

for genre in top_genres:
    df[f'genre_{genre.lower()}'] = df['genres'].str.contains(genre, case=False, na=False).astype(int)

print(f" Genres: Created binary features for {len(top_genres)} top genres")
print(f"  Top genres: {top_genres}")

# STEP 5: Extract temporal features
print("\nSTEP 5: Extract Temporal Features")
print("-" * 70)

df['release_decade'] = (df['title_year'] // 10 * 10).astype(int)
df['years_since_release'] = 2016 - df['title_year']

# One-hot encode decade
decade_dummies = pd.get_dummies(df['release_decade'], prefix='decade')
df = pd.concat([df, decade_dummies], axis=1)

print(f" Temporal features created:")
print(f"  • release_decade: {df['release_decade'].unique()}")
print(f"  • years_since_release: {df['years_since_release'].min():.0f} to {df['years_since_release'].max():.0f} years")

# STEP 6: Prepare features for modeling
print("\nSTEP 6: Prepare Features for Modeling")
print("-" * 70)

# Select features for the model
numerical_features = [
    'budget', 'duration', 'imdb_score', 'num_voted_users', 
    'cast_total_facebook_likes', 'director_facebook_likes',
    'log_budget', 'log_votes', 'budget_per_minute', 'budget_per_vote',
    'log_cast_popularity', 'log_director_popularity', 'cast_director_ratio',
    'quality_engagement', 'review_sentiment_score', 'years_since_release'
]

# Get categorical features (from one-hot encoding)
categorical_features = [col for col in df.columns if col.startswith('rating_') or 
                                                       col.startswith('genre_') or
                                                       col.startswith('decade_')]

all_features = numerical_features + categorical_features

print(f"Selected Features ({len(all_features)} total):")
print(f"  • Numerical features: {len(numerical_features)}")
print(f"  • Categorical features: {len(categorical_features)}")

# Create feature matrix and target
X = df[all_features].copy()
y = df['success'].copy()

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target distribution: {y.value_counts().to_dict()}")

# STEP 7: Handle any remaining missing values
print("\nSTEP 7: Handle Missing Values in Features")
print("-" * 70)

missing_before = X.isnull().sum().sum()
X = X.fillna(X.median(numeric_only=True))
X = X.fillna(0)
missing_after = X.isnull().sum().sum()

print(f"Missing values: {missing_before} → {missing_after}")

# STEP 8: Standardize features
print("\nSTEP 8: Standardize Features")
print("-" * 70)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

print(f" Features standardized (mean ≈ 0, std ≈ 1)")

# Save scaler for later use
with open('Ayebare_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print(f" Scaler saved as: Ayebare_scaler.pkl")

# ============================================================
# QUESTION 2: DEVELOP MACHINE LEARNING MODELS [15 MARKS]
# ============================================================

print("\n" + "="*70)
print("QUESTION 2: DEVELOP & COMPARE MODELS [15 MARKS]")
print("="*70)

# Split data
print("\nSTEP 1: Split Data into Train/Test Sets")
print("-" * 70)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_df, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test set: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nTarget distribution in training set:")
print(f"  • Class 0 (Unsuccessful): {(y_train==0).sum()} ({(y_train==0).mean()*100:.1f}%)")
print(f"  • Class 1 (Successful): {(y_train==1).sum()} ({(y_train==1).mean()*100:.1f}%)")

# MODEL 1: Logistic Regression
print("\n" + "-"*70)
print("MODEL 1: LOGISTIC REGRESSION")
print("-"*70)

print("Training Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_model.fit(X_train, y_train)

lr_train_pred = lr_model.predict(X_train)
lr_test_pred = lr_model.predict(X_test)
lr_test_proba = lr_model.predict_proba(X_test)[:, 1]

lr_metrics = {
    'Model': 'Logistic Regression',
    'Train Accuracy': accuracy_score(y_train, lr_train_pred),
    'Test Accuracy': accuracy_score(y_test, lr_test_pred),
    'Precision': precision_score(y_test, lr_test_pred),
    'Recall': recall_score(y_test, lr_test_pred),
    'F1-Score': f1_score(y_test, lr_test_pred),
    'ROC-AUC': roc_auc_score(y_test, lr_test_proba)
}

print(f" Logistic Regression trained")
print(f"  • Test Accuracy: {lr_metrics['Test Accuracy']:.4f}")
print(f"  • F1-Score: {lr_metrics['F1-Score']:.4f}")
print(f"  • ROC-AUC: {lr_metrics['ROC-AUC']:.4f}")

# MODEL 2: Random Forest
print("\n" + "-"*70)
print("MODEL 2: RANDOM FOREST CLASSIFIER")
print("-"*70)

print("Training Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=100, 
    max_depth=15, 
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

rf_train_pred = rf_model.predict(X_train)
rf_test_pred = rf_model.predict(X_test)
rf_test_proba = rf_model.predict_proba(X_test)[:, 1]

rf_metrics = {
    'Model': 'Random Forest',
    'Train Accuracy': accuracy_score(y_train, rf_train_pred),
    'Test Accuracy': accuracy_score(y_test, rf_test_pred),
    'Precision': precision_score(y_test, rf_test_pred),
    'Recall': recall_score(y_test, rf_test_pred),
    'F1-Score': f1_score(y_test, rf_test_pred),
    'ROC-AUC': roc_auc_score(y_test, rf_test_proba)
}

print(f" Random Forest trained")
print(f"  • Test Accuracy: {rf_metrics['Test Accuracy']:.4f}")
print(f"  • F1-Score: {rf_metrics['F1-Score']:.4f}")
print(f"  • ROC-AUC: {rf_metrics['ROC-AUC']:.4f}")

# MODEL COMPARISON
print("\n" + "-"*70)
print("MODEL COMPARISON")
print("-"*70)

comparison_df = pd.DataFrame([lr_metrics, rf_metrics])
print("\n" + comparison_df.to_string(index=False))

# Determine best model
best_model_choice = 'Random Forest' if rf_metrics['F1-Score'] > lr_metrics['F1-Score'] else 'Logistic Regression'
best_model = rf_model if best_model_choice == 'Random Forest' else lr_model
best_test_pred = rf_test_pred if best_model_choice == 'Random Forest' else lr_test_pred
best_test_proba = rf_test_proba if best_model_choice == 'Random Forest' else lr_test_proba

print(f"\nBEST MODEL: {best_model_choice}")
print(f"   Selected based on F1-Score: {max(rf_metrics['F1-Score'], lr_metrics['F1-Score']):.4f}")

# Feature Importance (Random Forest)
print("\n" + "-"*70)
print("FEATURE IMPORTANCE (Random Forest)")
print("-"*70)

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 15 Most Important Features:")
print(feature_importance.head(15).to_string(index=False))

# Confusion Matrix
print("\n" + "-"*70)
print("CONFUSION MATRIX - Best Model")
print("-"*70)

cm = confusion_matrix(y_test, best_test_pred)
print(f"\n{best_model_choice} Confusion Matrix:")
print(f"                 Predicted 0  Predicted 1")
print(f"Actual 0 (Unsuccessful):      {cm[0,0]:4d}       {cm[0,1]:4d}")
print(f"Actual 1 (Successful):        {cm[1,0]:4d}       {cm[1,1]:4d}")
print(f"\nInterpretation:")
print(f"  • True Negatives (TN): {cm[0,0]} - Correctly predicted unsuccessful films")
print(f"  • False Positives (FP): {cm[0,1]} - Predicted success but were unsuccessful")
print(f"  • False Negatives (FN): {cm[1,0]} - Predicted failure but were successful")
print(f"  • True Positives (TP): {cm[1,1]} - Correctly predicted successful films")

# STEP 9: Save best model
print("\n" + "-"*70)
print("SAVE BEST MODEL")
print("-"*70)

model_filename = 'Ayebare_best_model.pkl'
with open(model_filename, 'wb') as f:
    pickle.dump(best_model, f)

print(f"Best model saved as: {model_filename}")
print(f"   Model type: {best_model_choice}")
print(f"   Test Accuracy: {max(lr_metrics['Test Accuracy'], rf_metrics['Test Accuracy']):.4f}")

# ============================================================
# QUESTION 3: GENERATE PREDICTIONS [5 MARKS]
# ============================================================

print("\n" + "="*70)
print("QUESTION 3: GENERATE PREDICTIONS [5 MARKS]")
print("="*70)

print("\nGenerating predictions on test set...")

# Create predictions dataframe
predictions_df = pd.DataFrame({
    'actual_success': y_test.values,
    'predicted_success': best_test_pred,
    'probability_successful': best_test_proba,
    'prediction_correct': (y_test.values == best_test_pred).astype(int)
})

# Save predictions
pred_filename = 'Ayebare_predictions.csv'
predictions_df.to_csv(pred_filename, index=False)

print(f"Predictions saved as: {pred_filename}")
print(f"\nPrediction Summary:")
print(f"  • Total test samples: {len(predictions_df)}")
print(f"  • Correct predictions: {predictions_df['prediction_correct'].sum()}")
print(f"  • Incorrect predictions: {len(predictions_df) - predictions_df['prediction_correct'].sum()}")
print(f"  • Accuracy: {predictions_df['prediction_correct'].mean():.2%}")

print(f"\nSample predictions (first 10 rows):")
print(predictions_df.head(10).to_string())

# ============================================================
# QUESTION 4: MODEL INTERPRETATION [10 MARKS]
# ============================================================

print("\n" + "="*70)
print("QUESTION 4: MODEL INTERPRETATION [10 MARKS]")
print("="*70)

interpretation = f"""
MODEL PERFORMANCE ANALYSIS

1. EVALUATION METRICS SUMMARY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Best Model: {best_model_choice}

Key Metrics:
  • Accuracy: {max(lr_metrics['Test Accuracy'], rf_metrics['Test Accuracy']):.2%}
    - Proportion of correct predictions (successful + unsuccessful)
  
  • Precision: {max(lr_metrics['Precision'], rf_metrics['Precision']):.2%}
    - When we predict "successful", we're correct this % of time
    - Important for: Avoiding false marketing budgets on losing films
  
  • Recall: {max(lr_metrics['Recall'], rf_metrics['Recall']):.2%}
    - We catch this % of actual successful films
    - Important for: Not missing blockbuster opportunities
  
  • F1-Score: {max(lr_metrics['F1-Score'], rf_metrics['F1-Score']):.2%}
    - Balanced measure of precision and recall
  
  • ROC-AUC: {max(lr_metrics['ROC-AUC'], rf_metrics['ROC-AUC']):.2%}
    - Probability that model ranks a random successful film higher than
      a random unsuccessful film

2. CLASS-SPECIFIC PERFORMANCE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Confusion Matrix Analysis:
  • True Negatives (TN): {cm[0,0]} - Correctly identified box office flops
  • False Positives (FP): {cm[0,1]} - Films we thought would succeed but didn't
  • False Negatives (FN): {cm[1,0]} - Hidden blockbusters we missed
  • True Positives (TP): {cm[1,1]} - Correctly identified blockbusters

Classification Rate by Class:
  • Unsuccessful films correctly predicted: {cm[0,0]/(cm[0,0]+cm[0,1])*100:.1f}%
  • Successful films correctly predicted: {cm[1,1]/(cm[1,0]+cm[1,1])*100:.1f}%

3. SOURCES OF PREDICTION ERROR
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Type 1 Error (False Positives: {cm[0,1]}):
  Predicting success when film actually fails
  Causes:
    • Low-budget indie films with high IMDB scores (critical darlings)
    • Films with popular actors but weak marketing
    • Genre misclassification (awards bait vs commercial success)
  
Type 2 Error (False Negatives: {cm[1,0]}):
  Predicting failure when film actually succeeds
  Causes:
    • Unexpected viral successes (word-of-mouth blockbusters)
    • Star-driven films that exceed expectations
    • Franchise entries performing better than expected
    • Seasonal/holiday release advantages not captured

4. FEATURE IMPORTANCE INSIGHTS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Top Predictive Features (from Random Forest):
"""

for idx, row in feature_importance.head(5).iterrows():
    interpretation += f"\n  {idx+1}. {row['Feature']}: {row['Importance']:.4f}"

interpretation += f"""

Key Insights:
  • Budget is critical: Higher investment enables larger production/marketing
  • Quality matters: IMDB score and critical reception drive attendance
  • Audience engagement: Number of votes reflects reach and interest
  • Star power: Actor/director popularity influences expectations
  • Temporal trends: Release timing and decade affect success probability

5. PRACTICAL IMPLICATIONS FOR STUDIOS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Green-light Decisions:
  • Use model predictions as ONE input (not sole decision-making factor)
  • For high-uncertainty films (probability 45-55%), require additional expert review
  • Trust predictions most for films with clear patterns (>70% probability)

Marketing Allocation:
  • Films predicted as "low probability" should target niche audiences
  • Films predicted as "high probability" warrant wider release/marketing spend
  • Adjust distribution strategy based on confidence level

Risk Management:
  • {max(lr_metrics['Test Accuracy'], rf_metrics['Test Accuracy']):.1%} accuracy is good but not guaranteed
  • Use ensemble of models, not single model
  • Monitor predictions vs actuals to detect market shifts
  • Retrain model annually with new data

False Negative Cost: Very high (missing blockbusters)
  • Currently catching {cm[1,1]/(cm[1,0]+cm[1,1])*100:.1f}% of successful films
  • Consider lowering threshold to 40% probability to catch more hits
  
False Positive Cost: Moderate (wasting marketing budget)
  • Currently {cm[0,1]/(cm[0,0]+cm[0,1])*100:.1f}% of predicted successes fail
  • Accept this rate for better recall of actual successes

6. MODEL LIMITATIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Known Limitations:
  • Black swan events: Cannot predict pandemics, wars, cultural shifts
  • Release strategy not captured: Limited releases vs wide releases
  • Marketing spend unavailable: Major predictor missing from dataset
  • External events: Awards recognition, viral moments, competitor releases
  • Temporal decay: Box office market changes over 100-year period
  • Survivorship bias: Dataset may over-represent major studio films

Model Works Best For:
   Major studio releases (most data available)
   Films in well-established genres
   Films with significant budgets ($10M+)
   Films with strong cast/director recognition

Model May Struggle With:
  ✗ Ultra-low-budget indie films
  ✗ Experimental/niche films
  ✗ Franchise reboots (may be underrepresented)
  ✗ International films in non-English markets

7. RECOMMENDATIONS FOR IMPROVEMENT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Data Enhancements:
  • Add marketing budget: Critical missing feature
  • Add release strategy: Limited vs wide release distribution
  • Add audience demographics: Target age group and gender
  • Add trailer metrics: Views, sentiment from trailer comments
  • Add competition data: Other releases in same week/month

Model Improvements:
  • Ensemble methods: Combine predictions from multiple models
  • Deep learning: Neural networks for non-linear relationships
  • Temporal modeling: Account for market trends over time
  • Probabilistic calibration: Better confidence intervals
  • Explainability: SHAP values for per-film feature importance

Operational Improvements:
  • Monthly retraining: Adapt to shifting market
  • A/B testing: Compare model predictions vs actual decisions
  • Feedback loop: Track prediction accuracy by genre/year
  • Threshold optimization: Adjust confidence threshold based on business costs
"""

print(interpretation)

# Save interpretation
with open('Ayebare_model_interpretation.txt', 'w',encoding='utf-8') as f:
    f.write(interpretation)

print("\nModel interpretation saved as: Ayebare_model_interpretation.txt")

print("\n" + "="*70)
print("PART B COMPLETE!")
print("="*70)


PART B - FEATURE ENGINEERING, MODELING & PREDICTION

Loaded Ayebare.csv: 3288 rows, 30 columns
Target variable (success): {1: 1670, 0: 1618}

QUESTION 1: FEATURE ENGINEERING [10 MARKS]

STEP 1: Identify and Select Features
----------------------------------------------------------------------

FEATURE ENGINEERING PLAN:

1. NUMERICAL FEATURES (Direct Use):
   ─────────────────────────────────
   
   • budget: Production investment in millions USD
     Rationale: Higher budgets enable larger marketing and production scope
     
   • duration: Runtime in minutes
     Rationale: Affects audience reach (family films shorter, epics longer)
     Range: 80-180 minutes is optimal
     
   • imdb_score: Critical reception on 1-10 scale
     Rationale: Direct proxy for critical acclaim and quality perception
     
   • num_voted_users: Number of IMDB votes
     Rationale: Indicates audience engagement and film reach
     
   • cast_total_facebook_likes: Combined actor popularity
     Rationale: S